# job runner

> Executes functions in background threads with automatic tqdm progress capture.

In [ ]:
#| default_exp job_runner

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import threading
from typing import Callable, Any, Optional, Dict
from cjm_tqdm_capture.progress_info import ProgressInfo
from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.patch_tqdm import patch_tqdm

In [ ]:
#| export
class JobRunner:
    """
    Runs a callable in a background thread, patches tqdm inside the job,
    and forwards ProgressInfo updates to a ProgressMonitor under job_id.
    """
    def __init__(
        self,
        monitor: ProgressMonitor  # Progress monitor instance to receive updates
    ):
        "Initialize a job runner with a progress monitor"
        self.monitor = monitor
        self._threads: Dict[str, threading.Thread] = {}

    def start(
        self,
        job_id: str,  # Unique identifier for this job
        fn: Callable[..., Any],
        *args,
        patch_kwargs: Optional[Dict[str, Any]] = None,
        **kwargs
    ) -> threading.Thread:  # The thread running the job
        "Start a job in a background thread with automatic tqdm patching"
        patch_kwargs = patch_kwargs or {}
        def _target():
            "Internal thread target that wraps the job function with tqdm patching"
            def cb(
                info: ProgressInfo  # Progress update from the patched tqdm
            ):
                "Callback to forward progress updates to the monitor"
                self.monitor.update(job_id, info)
            try:
                with patch_tqdm(cb, **patch_kwargs):
                    fn(*args, **kwargs)
            except Exception as e:
                # Record an error as a final update
                self.monitor.update(job_id, ProgressInfo(progress=100.0, description=f"ERROR: {e}"))
        t = threading.Thread(target=_target, name=f"job-{job_id}", daemon=True)
        t.start()
        self._threads[job_id] = t
        return t

    def is_alive(
        self,
        job_id: str  # Unique identifier of the job to check
    ) -> bool:  # True if the job thread is still running
        "Check if a job's thread is still running"
        t = self._threads.get(job_id)
        return bool(t and t.is_alive())

    def join(
        self,
        job_id: str,  # Unique identifier of the job to wait for
        timeout: Optional[float] = None  # Maximum seconds to wait (None for indefinite)
    ) -> None:  # Returns when thread completes or timeout expires
        "Wait for a job's thread to complete"
        t = self._threads.get(job_id)
        if t:
            t.join(timeout)

In [ ]:
#| eval: false
from fastcore.test import test_eq, test_close
import uuid, time
from cjm_tqdm_capture.progress_monitor import ProgressMonitor

monitor = ProgressMonitor()
runner = JobRunner(monitor)

def demo_task():
    from tqdm import tqdm; import time
    for _ in tqdm(range(80), desc="JR A"):
        time.sleep(0.003)
    for _ in tqdm(range(20), desc="JR B"):
        time.sleep(0.003)

jid = str(uuid.uuid4())
runner.start(jid, demo_task, patch_kwargs=dict(min_delta_pct=5, min_update_interval=0.01, emit_initial=True))

deadline = time.time() + 5
while time.time() < deadline:
    snap = monitor.snapshot(jid)
    if snap and snap["completed"] and len(snap["bars"]) >= 2:
        break
    time.sleep(0.02)

snap = monitor.snapshot(jid)
assert snap is not None
test_eq(snap["completed"], True)
test_eq(len(snap["bars"]) >= 2, True)  # saw both bars
test_close(snap["overall_progress"], 100.0, 1e-6)
snap

JR B: 100%|██████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 327.30it/s]


{'start_time': 1755114979.8449705,
 'end_time': 1755114980.1538413,
 'completed': True,
 'bars': {'bar-1': ProgressInfo(progress=100.0, current=80, total=80, rate=None, elapsed='00:00', remaining=None, description='JR A', raw_output='', timestamp=1755114980.0919287, bar_id='bar-1', position=0),
  'bar-2': ProgressInfo(progress=100.0, current=20, total=20, rate=None, elapsed='00:00', remaining=None, description='JR B', raw_output='', timestamp=1755114980.1538398, bar_id='bar-2', position=0)},
 'latest': ProgressInfo(progress=100.0, current=20, total=20, rate=None, elapsed='00:00', remaining=None, description='JR B', raw_output='', timestamp=1755114980.1538398, bar_id='bar-2', position=0),
 'history': None,
 '_fallback_bar_id': None,
 'overall_progress': 100.0}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()